**FIN 585**  
**Diether**  
**Double Sort Portfolios**<br><br>

**1 Overview**

+ Goal $\rightarrow$ overview how to create double-sort portfolios

+ Double sort portfolios are very common in the academic literature, and also generally useful quant finance mmethod/tool.

+ Requires small extension of our standard coding tools $\rightarrow$ a three-way groupby instead of a two-way groupby.

+ Also cover some odds and ends about working with the CRSP data.

In [1]:
import numpy as np
import pandas as pd
from finance_byu.summarize import summary

<br>

**2. Raw CRSP Data**

+ Datafile $\rightarrow$ raw CRSP data in the feather format (very good format $\rightarrow$ compact and speedy).

+ Raw CRSP data contains negative prices.

+ If no transaction at the end of the trading, CRSP reports average quotes from market makers.

+ If quote based price $\rightarrow$ reported as a negative price in CRSP.

+ Typically researchers don't care about this distinction.

+ Typically just take the absolute value of price to solve problem.

In [2]:
df = pd.read_feather('12-mstk.ftr')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4889704 entries, 0 to 4889703
Data columns (total 12 columns):
 #   Column     Dtype         
---  ------     -----         
 0   permno     int64         
 1   caldt      datetime64[ns]
 2   cusip      str           
 3   ticker     str           
 4   shrcd      int64         
 5   excd       int64         
 6   siccd      int64         
 7   prc        float64       
 8   ret        float64       
 9   vol        float64       
 10  shr        float64       
 11  cumfacshr  float64       
dtypes: datetime64[ns](1), float64(5), int64(4), str(2)
memory usage: 496.8 MB


In [3]:
df[['prc','ret']].describe().round(3)

,prc,ret
count,4889704.000,4850170.000
mean,31.472,0.010
std,1785.113,0.171
min,-1925.000,-0.996
25%,2.875,-0.058
50%,12.750,0.000
75%,27.250,0.062
max,546725.000,24.000


In [4]:
df[['prc','ret']].quantile([0.05,0.1,0.15,0.20])

,prc,ret
0.05,-13.40625,-0.208958
0.10,-4.50000,-0.141509
0.15,-0.75000,-0.103448
0.20,1.28000,-0.077333


In [5]:
df['prc'] = df['prc'].abs()
df['me']  = df.eval("prc*shr/1000.0").where(df.eval("prc*shr > 1e-6"))

df[['prc','ret','me']].quantile([0.05,0.1,0.15,0.20])

,prc,ret,me
0.05,1.125,-0.208958,3.203125
0.10,2.125,-0.141509,6.084937
0.15,3.270,-0.103448,9.668750
0.20,4.625,-0.077333,14.250000


<br>

**3. Double Sort Portfolio Construction**

+ Sometimes you'll want to form portfolios based on two variables.

+ Example $\rightarrow$ forming based on lagged market-cap and momentum.<br><br>


**3.1 Breakpoints**

+ Need bins for both portfolio formation variables: momentum and market-cap.

+ Let's use NYSE breakpoints for market-cap.

+ We bin before splitting the sample so that the momentum breakpoints will be the same for both the small and large-cap stratification.

+ Called independent double sorting. $\leftarrow$ Fama French (1992)

+ Independent sorts make the comparisons across portfolio groupings more useful because the variation in momentum will be roughly the same across the portfolio groupings.

In [6]:
df['prclag'] = df.groupby('permno')['prc'].shift(1)
df['melag'] = df.groupby('permno')['me'].shift(1)

df['logret'] = df.eval("log(1+ret)")
df['mom'] = df.groupby('permno')['logret'].rolling(11).sum().reset_index(drop=True)
df['mom'] = df.groupby('permno')['mom'].shift(2)

+ **NYSE Breakpoint Function**

  + First wrote function in annually rebalanced market-cap portfolio notebook (merging application).

  + Take a look at the notebook to review.

In [7]:
def nyse_qcut(x,bp=[0.3,0.7]):
    bins = x.query("excd == 1")['melag'].quantile(bp).searchsorted(x['melag'])
    return pd.DataFrame(bins,index=x.index)

In [8]:
df = df.query("mom == mom and melag == melag and 10 <= shrcd <= 11 and "
              "prclag >= 5").reset_index(drop=True)

df['bins'] = df.groupby('caldt')['mom'].transform(pd.qcut,5,labels=False)

In [9]:
df['mebins'] = df.groupby('caldt',group_keys=False)[['excd','melag']].apply(nyse_qcut)
df

,permno,caldt,cusip,ticker,shrcd,excd,siccd,prc,ret,vol,shr,cumfacshr,me,prclag,melag,logret,mom,bins,mebins
0,10001,1987-02-27,39040610,GFGC,11,3,4920,6.25000,-0.074074,365.0,991.0,3.0,6.193750,6.75000,6.689250,-0.076961,0.196692,3,0
1,10001,1987-03-31,39040610,GFGC,11,3,4920,6.37500,0.036800,216.0,991.0,3.0,6.317625,6.25000,6.193750,0.036139,0.140122,2,0
2,10001,1987-04-30,39040610,GFGC,11,3,4920,6.12500,-0.039216,188.0,991.0,3.0,6.069875,6.37500,6.317625,-0.040006,0.038273,1,0
3,10001,1987-05-29,39040610,GFGC,11,3,4920,5.68750,-0.071429,211.0,991.0,3.0,5.636312,6.12500,6.069875,-0.074108,0.064560,2,0
4,10001,1987-06-30,39040610,GFGC,11,3,4920,5.87500,0.051429,146.0,991.0,3.0,5.822125,5.68750,5.636312,0.050150,0.034407,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2600412,93436,2023-08-31,88160R10,TSLA,11,3,9999,258.07999,-0.034962,25029170.0,3173994.0,1.0,819144.339780,267.42999,848821.183680,-0.035588,-0.126770,1,2
2600413,93436,2023-09-29,88160R10,TSLA,11,3,9999,250.22000,-0.030456,24395440.0,3179000.0,1.0,795449.380000,258.07999,819144.339780,-0.030929,-0.030128,1,2
2600414,93436,2023-10-31,88160R10,TSLA,11,3,9999,200.84000,-0.197346,25905681.0,3178921.0,1.0,638454.493640,250.22000,795449.380000,-0.219832,-0.027402,1,2
2600415,93436,2023-11-30,88160R10,TSLA,11,3,9999,240.08000,0.195379,26395792.0,3178921.0,1.0,763195.353680,200.84000,638454.493640,0.178463,0.095016,3,2


<br>

**3.2 Use Three Way Groupby to Double Sort**

+ Need to compute an equal-weight portfolio return for each date/market-cap/momentum bin combination.

+ Can accomplish that we a three-way groupby instead of the usual two-way.

+ Unstacking is a little bit trickier than usual.

In [10]:
port = df.groupby(['caldt','mebins','bins'])['ret'].mean()*100
port

caldt       mebins  bins
1927-01-31  0       0       -1.910607
                    1        4.403281
                    2        2.714714
                    3       -0.487830
                    4       -1.872933
                              ...    
2023-12-29  2       0       15.743082
                    1        7.981221
                    2        5.466751
                    3        5.499206
                    4        6.194155
Name: ret, Length: 17460, dtype: float64

In [11]:
port = port.unstack(level='bins')
port

bins                       0          1          2         3          4
caldt      mebins                                                      
1927-01-31 0       -1.910607   4.403281   2.714714 -0.487830  -1.872933
           1       -1.782547   3.929705   2.268072  1.915320   1.523350
           2       -2.710617   1.007208  -0.249842  0.482679   0.439077
1927-02-28 0        5.279374   8.702233   8.534500  7.696846   7.046855
           1        7.873242   8.262660   5.729551  5.466317   6.225053
...                      ...        ...        ...       ...        ...
2023-11-30 1        9.969049   7.284417   8.759236  8.720895  13.158676
           2       11.507685   6.864852   7.744622  9.799167  12.867838
2023-12-29 0       16.202035  13.818158  10.664346  2.530030  12.908388
           1       16.063112  13.736237  10.607317  8.929914   9.977076
           2       15.743082   7.981221   5.466751  5.499206   6.194155

[3492 rows x 5 columns]

In [12]:
port.query("mebins == 0")

,bins,0,1,2,3,4
caldt,mebins,,,,,
1927-01-31,0,-1.910607,4.403281,2.714714,-0.487830,-1.872933
1927-02-28,0,5.279374,8.702233,8.534500,7.696846,7.046855
1927-03-31,0,-3.161656,-0.143861,-3.005118,1.431260,-3.677485
1927-04-30,0,0.070524,0.127252,-2.497238,-0.205433,6.184415
1927-05-31,0,4.604330,9.898361,8.098779,8.364140,11.806871
...,...,...,...,...,...,...
2023-08-31,0,-7.656029,-4.905633,-2.342544,-5.126253,-4.635241
2023-09-29,0,-8.126687,-5.682505,-2.334717,-5.856552,-6.293907
2023-10-31,0,-9.843146,-4.129687,-5.066263,-4.720377,-7.652327


In [13]:
port = df.groupby(['caldt','mebins','bins'])['ret'].mean()*100
port = port.unstack(level=['mebins','bins'])
port

mebins              0                                                     1  \
bins                0          1          2         3          4          0   
caldt                                                                         
1927-01-31  -1.910607   4.403281   2.714714 -0.487830  -1.872933  -1.782547   
1927-02-28   5.279374   8.702233   8.534500  7.696846   7.046855   7.873242   
1927-03-31  -3.161656  -0.143861  -3.005118  1.431260  -3.677485  -5.050995   
1927-04-30   0.070524   0.127252  -2.497238 -0.205433   6.184415  -3.122029   
1927-05-31   4.604330   9.898361   8.098779  8.364140  11.806871   4.265021   
...               ...        ...        ...       ...        ...        ...   
2023-08-31  -7.656029  -4.905633  -2.342544 -5.126253  -4.635241  -7.461510   
2023-09-29  -8.126687  -5.682505  -2.334717 -5.856552  -6.293907  -8.290818   
2023-10-31  -9.843146  -4.129687  -5.066263 -4.720377  -7.652327  -8.019190   
2023-11-30   9.634220   8.276349   5.000714  4.706621  11.464658   9.969049   
2023-12-29  16.202035  13.818158  10.664346  2.530030  12.908388  16.063112   

mebins                                                         2            \
bins                1          2         3          4          0         1   
caldt                                                                        
1927-01-31   3.929705   2.268072  1.915320   1.523350  -2.710617  1.007208   
1927-02-28   8.262660   5.729551  5.466317   6.225053   7.398887  4.747800   
1927-03-31  -4.425645  -1.489674 -1.012519   0.136754  -5.106712 -5.163292   
1927-04-30  -0.392742   1.244412 -0.305769   3.505479  -8.523350 -1.267781   
1927-05-31   5.448355  10.872957  6.904446   8.227843   2.701400  2.274668   
...               ...        ...       ...        ...        ...       ...   
2023-08-31  -6.153799  -3.828613 -2.645074  -3.886169  -6.478332 -4.210187   
2023-09-29  -5.977511  -3.836350 -4.513544  -7.181078  -5.480524 -4.608046   
2023-10-31  -6.850645  -4.369744 -5.716230  -6.784115  -8.873822 -3.482565   
2023-11-30   7.284417   8.759236  8.720895  13.158676  11.507685  6.864852   
2023-12-29  13.736237  10.607317  8.929914   9.977076  15.743082  7.981221   

mebins                                     
bins               2         3          4  
caldt                                      
1927-01-31 -0.249842  0.482679   0.439077  
1927-02-28  3.792231  3.228853   4.004888  
1927-03-31 -1.066466  1.081037   1.190557  
1927-04-30  0.567102  0.194981   1.186419  
1927-05-31  4.798948  7.005733   7.412739  
...              ...       ...        ...  
2023-08-31 -2.587123 -2.224414  -2.067464  
2023-09-29 -4.226064 -4.009363  -5.880407  
2023-10-31 -3.570412 -3.472790  -5.632417  
2023-11-30  7.744622  9.799167  12.867838  
2023-12-29  5.466751  5.499206   6.194155  

[1164 rows x 15 columns]

In [14]:
summary(port).loc[['mean','std','tstat']].round(3)

mebins      0                                  1                              \
bins        0      1      2      3      4      0      1      2      3      4   
mean    0.571  1.075  1.296  1.426  1.669  0.534  0.927  1.059  1.214  1.542   
std     8.374  7.149  6.703  6.420  7.495  8.603  6.849  6.185  5.922  6.848   
tstat   2.325  5.128  6.595  7.580  7.596  2.119  4.620  5.840  6.994  7.681   

mebins      2                              
bins        0      1      2      3      4  
mean    0.514  0.814  0.889  1.074  1.323  
std     7.970  6.275  5.499  5.401  6.125  
tstat   2.202  4.427  5.517  6.787  7.369